In [ ]:
import pandas as pd
from statsbombpy import sb
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.preprocessing import LabelEncoder
import os
import subprocess
import json
import webbrowser
import warnings

warnings.filterwarnings('ignore')

# ==========================================
# 1. DIRECTORY SETUP
# ==========================================
# Since main.ipynb is in the 'notebooks' folder, paths are relative to it.
RAW_DIR = "../data/raw/"
CLEAN_DIR = "../data/clean/"
EDA_DIR = "eda_labs/"  # New subfolder for generated EDA notebooks

os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(CLEAN_DIR, exist_ok=True)
os.makedirs(EDA_DIR, exist_ok=True)

def run_cmd(cmd):
    """Executes terminal commands and returns the output."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    return result.stdout + result.stderr

# ==========================================
# 2. UI: MAIN NAVIGATION MENU
# ==========================================
out_main = widgets.Output()

menu_tabs = widgets.ToggleButtons(
    options=['📥 1. Download Data', '🧹 2. EDA & Cleaning', '🚀 3. Streamlit', '🐙 4. Git Manager'],
    description='🎯 Action:',
    button_style='info'
)

# ==========================================
# 3. MODULE 1: DATA DOWNLOAD CENTER
# ==========================================
print("⏳ Initializing... fetching competitions from StatsBomb API...")
df_comps = sb.competitions()
comp_names = sorted(df_comps['competition_name'].unique().tolist())
clear_output()

dd_comp = widgets.Dropdown(options=comp_names, description='🏆 Comp:')
dd_season = widgets.Dropdown(options=[], description='📅 Season:')
rb_mode = widgets.RadioButtons(options=['Single Match', 'Full Season Bulk'], value='Single Match', description='⚙️ Mode:')
dd_match = widgets.Dropdown(options=[], description='⚽ Match:', layout=widgets.Layout(width='80%'))
btn_download = widgets.Button(description='🚀 DOWNLOAD DATA', button_style='danger')
out_download = widgets.Output()

def on_comp_change(*args):
    if dd_comp.value:
        seasons = df_comps[df_comps['competition_name'] == dd_comp.value]['season_name'].unique().tolist()
        dd_season.options = sorted(seasons, reverse=True)

def on_season_mode_change(*args):
    if dd_comp.value and dd_season.value:
        row = df_comps[(df_comps['competition_name'] == dd_comp.value) & (df_comps['season_name'] == dd_season.value)].iloc[0]
        if rb_mode.value == 'Single Match':
            dd_match.disabled = False
            try:
                matches = sb.matches(competition_id=row['competition_id'], season_id=row['season_id'])
                dd_match.options = [(f"{r['match_date']} | {r['home_team']} vs {r['away_team']}", r['match_id']) for _, r in matches.iterrows()]
            except:
                dd_match.options = [('No data available', None)]
        else:
            dd_match.disabled = True
            dd_match.options = [('Full Season selected', None)]

def execute_download(b):
    with out_download:
        clear_output()
        print("⏳ Download initiated, please wait...")
        row = df_comps[(df_comps['competition_name'] == dd_comp.value) & (df_comps['season_name'] == dd_season.value)].iloc[0]
        c_name, s_name = dd_comp.value.replace(" ", "_").lower(), dd_season.value.replace("/", "_")
        try:
            if rb_mode.value == 'Single Match' and dd_match.value:
                df = sb.events(match_id=dd_match.value)
                fname = f"statsbomb_{c_name}_{s_name}_match_{dd_match.value}.csv"
            else:
                df_matches = sb.matches(competition_id=row['competition_id'], season_id=row['season_id'])
                df = pd.concat([sb.events(match_id=m['match_id']) for _, m in df_matches.iterrows()], ignore_index=True)
                fname = f"statsbomb_{c_name}_{s_name}_full.csv"
            
            path = os.path.join(RAW_DIR, fname)
            df.to_csv(path, index=False)
            print(f"✅ SUCCESS! Data saved to: {path}")
            update_raw_list()
        except Exception as e:
            print(f"❌ ERROR: {e}")

dd_comp.observe(on_comp_change, names='value')
dd_season.observe(on_season_mode_change, names='value')
rb_mode.observe(on_season_mode_change, names='value')
btn_download.on_click(execute_download)

ui_download = widgets.VBox([
    widgets.HTML("<h3>📥 Module 1: Data Download Center</h3>"),
    dd_comp, dd_season, rb_mode, dd_match, btn_download, out_download
])

# ==========================================
# 4. MODULE 2: EDA & CLEANING ENVIRONMENT
# ==========================================
dd_raw_files = widgets.Dropdown(description='📂 Raw File:', layout=widgets.Layout(width='60%'))
btn_clean = widgets.Button(description='🏗️ GENERATE EDA NOTEBOOK', button_style='success', icon='flask')
out_clean = widgets.Output()

def update_raw_list():
    files = [f for f in os.listdir(RAW_DIR) if f.endswith('.csv')]
    dd_raw_files.options = files if files else ['No files found in data/raw/']

def generate_eda_notebook(b):
    with out_clean:
        clear_output()
        if not dd_raw_files.value or dd_raw_files.value.startswith('No'):
            print("❌ Select a valid raw file from the dropdown.")
            return
            
        selected_file = dd_raw_files.value
        
        # 1. Sanitization (Apostrophes & Spaces)
        safe_file_path = selected_file.replace("'", "\\'") 
        sql_table_name = selected_file.replace('.csv', '_clean').replace("'", "").replace(" ", "_").lower()
        safe_notebook_name = selected_file.replace('.csv', '').replace("'", "")
        new_notebook_name = f"EDA_{safe_notebook_name}.ipynb"
        
        template_path = "EDA_Template.ipynb"
        
        # 2. Check if the template exists
        if not os.path.exists(template_path):
            print(f"❌ ERROR: Master template '{template_path}' not found in the current folder.")
            print("Please ensure you have created it.")
            return
            
        print(f"⚙️ Reading Master Template and generating '{new_notebook_name}'...")
        
        # 3. Read Template -> Replace placeholders -> Save to eda_labs/
        with open(template_path, 'r', encoding='utf-8') as file:
            template_data = file.read()
            
        # Inject the dynamic filenames into the template
        template_data = template_data.replace("{{FILE_NAME}}", safe_file_path)
        template_data = template_data.replace("{{TABLE_NAME}}", sql_table_name)
        
        # Save the new customized notebook
        notebook_path = os.path.join(EDA_DIR, new_notebook_name)
        with open(notebook_path, 'w', encoding='utf-8') as file:
            file.write(template_data)
            
        print(f"✅ EDA ENVIRONMENT CREATED SUCCESSFULLY!")
        print(f"👉 Navigate to the 'eda_labs/' folder in the sidebar and open '{new_notebook_name}' to begin your analysis.")

btn_clean.on_click(generate_eda_notebook)

ui_clean = widgets.VBox([
    widgets.HTML("<h3>🧹 Module 2: EDA & Data Cleaning Laboratory</h3>"),
    widgets.HTML("<i>Select a raw dataset. A dedicated Jupyter Notebook containing your statistical tools will be generated.</i>"),
    dd_raw_files, btn_clean, out_clean
])

# ==========================================
# 5. MODULE 3: STREAMLIT DASHBOARD
# ==========================================
ui_streamlit = widgets.VBox([
    widgets.HTML("<h3>🚀 Module 3: Streamlit Presentation Layer</h3>"),
    widgets.HTML("<p>Your polished data is securely stored in PostgreSQL.</p>"),
    widgets.HTML("<p>To view the dashboard, open the VS Code integrated terminal and run:</p>"),
    widgets.HTML("<code>streamlit run app.py</code>")
])

# ==========================================
# 6. MODULE 4: GIT MANAGER
# ==========================================
txt_commit = widgets.Text(description='📝 Message:', placeholder='Enter commit message...')
btn_status = widgets.Button(description='🔍 Check Status', button_style='info')
btn_pull = widgets.Button(description='⬇️ Pull Updates', button_style='warning')
btn_push = widgets.Button(description='⬆️ Add + Commit + Push', button_style='success')
btn_browser = widgets.Button(description='🌐 Open in Browser', button_style='primary')
out_git = widgets.Output()

def git_status(b):
    with out_git:
        clear_output()
        print("🔍 Checking repository status...")
        print(run_cmd("git status"))

def git_pull(b):
    with out_git:
        clear_output()
        print("⬇️ Pulling latest updates from GitHub...")
        print(run_cmd("git pull"))

def git_push(b):
    with out_git:
        clear_output()
        commit_msg = txt_commit.value if txt_commit.value else "Routine update via Orchestrator"
        print("⬆️ Pushing to GitHub...")
        
        # 1. Add all changes (including deleted files)
        print("1. Staging files...")
        add_result = run_cmd("git add -A")
        print(add_result)
        
        # 2. Commit
        print(f"2. Committing with message: '{commit_msg}'...")
        commit_result = run_cmd(f'git commit -m "{commit_msg}"')
        print(commit_result)
        
        # 3. Explicitly push to origin main (forces the connection)
        print("3. Pushing to remote repository (origin main)...")
        push_result = run_cmd("git push origin main")
        print(push_result)
        
        # 4. Clear the text box for the next time
        txt_commit.value = ""
        print("✅ Operation Completed! (Note: VS Code Source Control panel might take a few seconds to update visually).")

def open_github(b):
    with out_git:
        clear_output()
        print("🌐 Retrieving remote URL...")
        url_raw = run_cmd("git config --get remote.origin.url").strip()
        
        if not url_raw:
            print("❌ No remote repository configured.")
            return
            
        if url_raw.startswith("git@github.com:"):
            url = url_raw.replace("git@github.com:", "https://github.com/").replace(".git", "")
        elif url_raw.endswith(".git"):
            url = url_raw.replace(".git", "")
        else:
            url = url_raw
            
        print(f"🔗 Opening: {url}")
        webbrowser.open(url)

btn_status.on_click(git_status)
btn_pull.on_click(git_pull)
btn_push.on_click(git_push)
btn_browser.on_click(open_github)

ui_git = widgets.VBox([
    widgets.HTML("<h3>🐙 Module 4: Git Synchronization Manager</h3>"),
    widgets.HBox([btn_status, btn_pull, btn_browser]),
    widgets.HTML("<br>"),
    widgets.HBox([txt_commit, btn_push]),
    out_git
])

# ==========================================
# 7. NAVIGATION CONTROLLER
# ==========================================
def handle_navigation(change):
    with out_main:
        clear_output()
        if change['new'].startswith('📥'):
            on_comp_change()
            on_season_mode_change()
            display(ui_download)
        elif change['new'].startswith('🧹'):
            update_raw_list()
            display(ui_clean)
        elif change['new'].startswith('🚀'):
            display(ui_streamlit)
        elif change['new'].startswith('🐙'):
            display(ui_git)

menu_tabs.observe(handle_navigation, names='value')

# Initialize Startup View
display(menu_tabs, out_main)
handle_navigation({'new': menu_tabs.value})

ToggleButtons(button_style='info', description='🎯 Action:', options=('📥 1. Download Data', '🧹 2. EDA & Cleanin…

Output()